# 🎲 Multiple Testing
**ISLP Chapter 13 · Pattern Recognition for the Rest of Us**

> Test 100 hypotheses at α=0.05 and you expect 5 false positives just by chance — even if nothing is real. Multiple testing procedures control this inflation of false discoveries.

### What you'll learn
- The multiple testing problem: why p-values get inflated
- Family-wise error rate (FWER) and Bonferroni correction
- False discovery rate (FDR) and the Benjamini-Hochberg procedure
- When to use FWER vs FDR
- Practical applications: genomics, A/B testing, feature selection

### Time: ~45 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
print('✓ Setup complete')
from statsmodels.stats.multitest import multipletests
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
print('✓ Setup complete')

## 🎯 Part 1 — The Problem: False Positives at Scale

At α=0.05, each test has a 5% chance of a false positive. If we run m independent tests, the probability of *at least one* false positive is:

```
P(at least one false positive) = 1 - (1-0.05)^m
```

For m=100 tests: 1 - 0.95^100 ≈ 99.4% chance of at least one false positive!

In [ ]:
# Simulate: false positive inflation
m_values = [1, 5, 10, 20, 50, 100, 200, 500, 1000]
alpha = 0.05
fwer = [1 - (1-alpha)**m for m in m_values]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(m_values, fwer, 'o-', color='#e85d20', lw=2.5, markersize=7)
ax.axhline(0.05, color='#888', lw=1, ls='--', label='Single test α=0.05')
ax.set_xlabel('Number of tests (m)')
ax.set_ylabel('Probability of at least one false positive')
ax.set_xscale('log')
ax.set_title('Multiple Testing Problem: FWER Inflation')
ax.legend()
plt.tight_layout(); plt.show()
print('📌 With 100 tests at α=0.05, there is a 99.4% chance of at least one false positive!')
print('   Bonferroni or BH correction is needed')

In [ ]:
# Concrete simulation
np.random.seed(42)
m = 100  # tests
n = 30   # observations per test

# All 100 null hypotheses are TRUE (no real effect)
# Any rejection is a false positive
p_values_null = [stats.ttest_ind(
    np.random.normal(0,1,n), np.random.normal(0,1,n)
).pvalue for _ in range(m)]

fig, ax = plt.subplots(figsize=(10, 3))
colors = ['#e85d20' if p < 0.05 else '#1e5fa8' for p in p_values_null]
ax.bar(range(m), sorted(p_values_null), color=[c for _,c in sorted(zip(p_values_null, colors))],
      edgecolor='white', width=0.9)
ax.axhline(0.05, color='black', lw=1.5, ls='--', label='α=0.05')
fp = sum(p < 0.05 for p in p_values_null)
ax.set_title(f'100 tests, ALL null true — {fp} false positives (orange) at α=0.05')
ax.set_xlabel('Test (sorted by p-value)'); ax.set_ylabel('p-value')
ax.legend()
plt.tight_layout(); plt.show()
print(f'False positives: {fp} out of {m} tests (expected: {m*0.05:.0f})')

In [ ]:
# Bonferroni vs BH correction
# Now: 80 true nulls, 20 real effects
np.random.seed(1)
m_null = 80; m_alt = 20; n = 30
p_null = [stats.ttest_ind(np.random.normal(0,1,n), np.random.normal(0,1,n)).pvalue for _ in range(m_null)]
p_alt  = [stats.ttest_ind(np.random.normal(0,1,n), np.random.normal(2,1,n)).pvalue for _ in range(m_alt)]
p_all  = np.array(p_null + p_alt)
true_null = np.array([True]*m_null + [False]*m_alt)

results_corr = {}
for method, label in [('bonferroni','Bonferroni (FWER)'), ('fdr_bh','Benjamini-Hochberg (FDR)')]:
    reject, p_adj, _, _ = multipletests(p_all, alpha=0.05, method=method)
    tp = np.sum(reject & ~true_null)  # true positives
    fp = np.sum(reject & true_null)   # false positives
    fn = np.sum(~reject & ~true_null) # false negatives
    results_corr[label] = {'Rejected': reject.sum(), 'True Pos': tp, 'False Pos': fp, 'False Neg': fn, 'Power': tp/m_alt}
    print(f'{label}:')
    print(f'  Rejected: {reject.sum()} | TP={tp} | FP={fp} | FN={fn} | Power={tp/m_alt:.2f}')

print('\n📌 Bonferroni is very conservative — avoids false positives but misses many real effects')
print('   BH-FDR is more powerful — accepts some false positives to find more true effects')

In [ ]:
# Visualize: BH procedure step by step
sorted_p = np.sort(p_all)
m_total = len(p_all)
bh_threshold = np.arange(1, m_total+1) * 0.05 / m_total

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(range(1, m_total+1), sorted_p, s=20, alpha=0.6, label='p-values (sorted)',
          c=['#1a7a45' if p < 0.05 else '#888' for p in sorted(p_all)])
ax.plot(range(1, m_total+1), bh_threshold, color='#e85d20', lw=2, label='BH threshold k×α/m')
bonf_threshold = 0.05/m_total
ax.axhline(bonf_threshold, color='#c0392b', lw=1.5, ls='--', label=f'Bonferroni α/m={bonf_threshold:.4f}')
bh_reject_idx = np.where(sorted_p <= bh_threshold)[0]
if len(bh_reject_idx): ax.axvline(bh_reject_idx[-1]+1, color='#e85d20', lw=1, ls=':')
ax.set_xlabel('Rank (k)'); ax.set_ylabel('p-value')
ax.set_title('Benjamini-Hochberg Procedure: Reject all p ≤ k×α/m up to largest k')
ax.legend(); ax.set_ylim(0, 0.2)
plt.tight_layout(); plt.show()

In [ ]:
# Exercise workspace
# Task 1: Load a real genomics-style dataset and apply multiple testing correction
# Simulate 1000 gene expression comparisons with 50 truly differentially expressed genes
np.random.seed(42)
m_genes = 1000; m_de = 50
p_genes_null = [stats.ttest_ind(np.random.normal(0,1,20),np.random.normal(0,1,20)).pvalue for _ in range(m_genes-m_de)]
p_genes_de   = [stats.ttest_ind(np.random.normal(0,1,20),np.random.normal(1.5,1,20)).pvalue for _ in range(m_de)]
p_genes_all  = np.array(p_genes_null + p_genes_de)
# YOUR CODE HERE: apply Bonferroni and BH. Compare TP, FP, power.

# Task 2: A/B testing scenario — you tested 20 website variants simultaneously
# p-values from each test are given below. Which pass Bonferroni? Which pass BH at q=0.05?
ab_pvalues = [0.001, 0.023, 0.041, 0.087, 0.12, 0.18, 0.002, 0.045, 0.22, 0.008,
              0.19, 0.31, 0.005, 0.11, 0.67, 0.043, 0.28, 0.0003, 0.55, 0.032]
# YOUR CODE HERE

In [ ]:
answers = {
    'q1': '',  # What is FWER and what does Bonferroni control?
    'q2': '',  # What is FDR and what does Benjamini-Hochberg control?
    'q3': '',  # When would you choose FDR correction over FWER correction?
    'q4': '',  # With 1000 tests at α=0.05, how many false positives do you expect without correction?
    'q5': '',  # What is the Bonferroni correction threshold for 50 tests at α=0.05?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print('❌ Still empty: '+str(missing) if missing else '✅ Done! File → Save a copy in GitHub')

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Multiple Testing"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()


## 📚 Further Reading
- [ISLP Ch 13](https://intro-stat-learning.github.io/ISLP/) — Multiple Testing
- [statsmodels multipletests](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html)
- [Cross-Validation notebook](cross-validation.ipynb) — another way to control model selection error

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*